## 1. Start

In [1]:
import pandas as pd
import numpy as np
import csv
assert csv  # silence pyflakes
import xlsxwriter
assert xlsxwriter  # silence pyflakes
import xlrd
assert xlrd  # silence pyflakes
import math
assert math  # silence pyflakes
import nbimporter
import traceback
from pathlib import Path
import os

In [2]:
try:
    libraries_path = Path(__file__).resolve()
except NameError:
    libraries_path = Path(os.getcwd()) / "Libraries_pd.ipynb"
    if not libraries_path.exists():
        libraries_path = Path(os.getcwd())

Libraries_dir = libraries_path.parent

print(f"📁 Libraries_pd.ipynb — directory impostata a: {Libraries_dir}")


📁 Libraries_pd.ipynb — directory impostata a: c:\Users\Saverio_admin\Documents\NEST_py


# 2. Files reading

In [ ]:
# ✅ 

base_path = Libraries_dir
root_path = os.path.join(base_path, "Energy_systems_pd")
    
EndOfLifeInputParameters_path = base_path / 'EndOfLifeInputParameters.xlsx'

# Check if 'data_input_filename' already exists in the global scope
try:
    data_input_filename
except NameError:
    # It doesn't exist yet, so ask for it (this happens the first time)
    data_input_filename = input("Enter the DataInput filename, namely the name of the file for your foreground scenario (e.g. DataInput_S1.xlsm): ").strip()
else:
    # It already exists (e.g. defined in RUN_pd.ipynb or a previous notebook run)
    # Pass and keep the existing value
    print(f"Using existing filename: {data_input_filename}")

generatorsConfig_path = base_path / data_input_filename
energysystem_path = base_path / data_input_filename     # used by the Energy_systems_pd notebooks
Scenario_path = base_path / data_input_filename

EndOfLifeInputParameters = pd.read_excel(EndOfLifeInputParameters_path, sheet_name='EndOfLifeInputParameters')
generatorsConfig = pd.read_excel(generatorsConfig_path, sheet_name='generatorsConfig')
Scenario = pd.read_excel(Scenario_path, sheet_name='Scenario')

In [ ]:
boiler_path = os.path.join(root_path, 'Boiler.ipynb')
Biomassboiler_path = os.path.join(root_path, 'BiomassBoiler.ipynb')
Fueloilboiler_path = os.path.join(root_path, 'FuelOilBoiler.ipynb')
EHP_path = os.path.join(root_path, 'ElectricHeatPump.ipynb')
GEHP_path = os.path.join(root_path, 'GasEngineHeatPump.ipynb')
GAHP_path = os.path.join(root_path, 'GasAbsorptionHeatPump.ipynb')
Geothermal_path = os.path.join(root_path, 'GeothermalPlant.ipynb')
RICE_path = os.path.join(root_path, 'CombinedHeatPower_InternalCombustionEngine.ipynb')
ORC_path = os.path.join(root_path, 'CombinedHeatPower_OrganicRankineCycle.ipynb')
WtE_path = os.path.join(root_path, 'CombinedHeatPower_WasteToEnergy.ipynb')
CombinedCycle_path = os.path.join(root_path, 'CombinedHeatPower_CombinedCycle.ipynb')
GasTurbine_path = os.path.join(root_path, 'CombinedHeatPower_Turbogas.ipynb')
HeatRecovery_path = os.path.join(root_path, 'HeatRecovery.ipynb')
SolarThermal_path = os.path.join(root_path, 'SolarThermal.ipynb')
ThermalConcreteStorage_path = os.path.join(root_path, 'ThermalConcreteStorage.ipynb')
ThermalSteelStorage_path = os.path.join(root_path, 'ThermalSteelStorage.ipynb')
AbsorptionChiller_path = os.path.join(root_path, 'CoolingAbsorptionChiller.ipynb')
CompressionChiller_path = os.path.join(root_path, 'CoolingCompressionChiller.ipynb')
DryCooler_path = os.path.join(root_path, 'CoolingDryCooler.ipynb')
CoolingTower_path = os.path.join(root_path, 'CoolingEvaporativeTower.ipynb')
# Auxiliaries_path = os.path.join(root_path, 'Auxiliaries.ipynb')
Appliances_path = os.path.join(root_path, 'Appliances.ipynb')

# 3. General Scenario information

In [ ]:
RSP = generatorsConfig.loc[generatorsConfig['Information'] == 'Reference Study Period (RSP) [Year]', 'Data'].values[0]
# ReqSL = generatorsConfig.loc[generatorsConfig['Information'] == 'Required Service Life (ReqSL) [Year]', 'Data'].values[0]
# Losses = generatorsConfig.loc[generatorsConfig['Information'] == 'Distribution losses', 'Data'].values[0]

Bd_Name = Scenario['Name']
Bd_Area = Scenario['Zone net floor area [m2]']
Bd_Use = Scenario['End Use']
Heating_systems = Scenario['Heating Energy Systems']
Cooling_systems = Scenario['Cooling Energy Systems']

# 4. End-of-Life parameters import

In [6]:
#End-of-life parameters
LandfillMetals2 = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of metals (e.g low-alloyed steel) sent to recycling at end of life', 'value'].values[0])
LandfillMetals = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of metals  (e.g low-alloyed steel) sent to landfill at end of life', 'value'].values[0])
LandfillElastomer = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of elastomers sent to landfill at end of life', 'value'].values[0])
LandfillPVC = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of PVC sent to landfill at end of life', 'value'].values[0])
LandfillPEHD = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of high-density polyethylene (HDPE) sent to landfill at end of life', 'value'].values[0])
LandfillRefrigerant = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of refrigerant gases sent to landfill at end of life', 'value'].values[0])
LandfillElectronics = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of electronic waste sent to landfill at end of life', 'value'].values[0])
LandfillClaybrick = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of clay bricks sent to landfill at end of life', 'value'].values[0])
LandfillConcrete = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of concrete sent to landfill at end of life', 'value'].values[0])
LandfillGlass = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of glass sent to landfill at end of life', 'value'].values[0])
LandfillMineralWool = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of mineral wool sent to landfill at end of life', 'value'].values[0])
LandfillBrass = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of brass sent to landfill at end of life', 'value'].values[0])
LandfillCardboard = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of cardboard sent to landfill at end of life', 'value'].values[0])
LandfillInert = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of inert materials sent to landfill at end of life', 'value'].values[0])
LandfillPUfoam = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of polyurethane foam sent to landfill at end of life', 'value'].values[0])
LandfillPP = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of polypropylene (PP) sent to landfill at end of life', 'value'].values[0])

IncinerationElastomer = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of elastomers sent to incineration at end of life', 'value'].values[0])
IncinerationPVC = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of PVC sent to incineration at end of life', 'value'].values[0])
IncinerationElectronics = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of electronic waste sent to incineration at end of life', 'value'].values[0])
IncinerationLubricatingOil = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of lubricating oil sent to incineration at end of life', 'value'].values[0])
IncinerationHazardous = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of hazardous waste sent to incineration at end of life', 'value'].values[0])
IncinerationPEHD = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of high-density polyethylene (HDPE) sent to incineration at end of life', 'value'].values[0])
IncinerationPEfoam = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of polyethylene foam sent to incineration at end of life', 'value'].values[0])
IncinerationPP = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of polypropylene (PP) sent to incineration at end of life', 'value'].values[0])
IncinerationEPS = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of expanded polystyrene (EPS) sent to incineration at end of life', 'value'].values[0])
IncinerationCardboard = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of cardboard sent to incineration at end of life', 'value'].values[0])
IncinerationPUfoam = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of polyurethane foam sent to incineration at end of life', 'value'].values[0])

#Materials that goes recycling
RecyclingMetals2 = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of special metals (e.g. stainless steel, chromium steel) sent to recycling at end of life', 'value'].values[0])
RecyclingMetals = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of metals (e.g low-alloyed steel) sent to recycling at end of life', 'value'].values[0])
RecyclingPEHD = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of high-density polyethylene (HDPE) sent to recycling at end of life', 'value'].values[0])
RecyclingConcrete = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of concrete sent to recycling at end of life', 'value'].values[0])
RecyclingClaybrick = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of clay bricks sent to recycling at end of life', 'value'].values[0])
RecyclingCardboard = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Share of cardboard sent to recycling at end of life', 'value'].values[0])

# Recycling efficiency process
RecyclingEfficiencyMetals = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of metals', 'value'].values[0])
RecyclingEfficiencyPlastics = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of plastics', 'value'].values[0])
RecyclingEfficiencyConcrete = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of concrete', 'value'].values[0])
RecyclingEfficiencyGlass = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of glass', 'value'].values[0])
RecyclingEfficiencyClaybrick = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of clay bricks', 'value'].values[0])
RecyclingEfficiencyMineralWool = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of mineral wool', 'value'].values[0])
RecyclingEfficiencyCardboard = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycling efficiency of cardboard', 'value'].values[0])

#Recycled content materials
RecycledContentMetals = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of steel (fraction of total steel mass)', 'value'].values[0])
RecycledContentPlastic = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of plastics (fraction of total plastic mass)', 'value'].values[0])
RecycledContentClaybrick = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of clay bricks', 'value'].values[0])
RecycledContentConcrete = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of concrete', 'value'].values[0])
RecycledContentCardboard = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of cardboard', 'value'].values[0])
RecycledContentMineralWool = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of mineral wool', 'value'].values[0])
RecycledContentGlass = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Recycled content of glass', 'value'].values[0])


#Substitution ration materials
SRmetals = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Material substitution ratio for metals', 'value'].values[0])
SRinert = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Material substitution ratio for inert materials', 'value'].values[0])
SRclaybrick = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Material substitution ratio for lightweight bricks', 'value'].values[0])

#Lower Heating Values materials for End-of-Life
LHVElastomer = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of elastomers', 'value'].values[0])
LHVElectronics = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of electronic waste', 'value'].values[0])
LHVHazardous = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of hazardous waste', 'value'].values[0])
LHVHDPE = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of high-density polyethylene (HDPE)', 'value'].values[0])
LHVPEfoam = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of polyethylene foam', 'value'].values[0])
LHVPP = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of polypropylene (PP)', 'value'].values[0])
LHVPVC = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of PVC', 'value'].values[0])
LHVEPS = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of expanded polystyrene (EPS)', 'value'].values[0])
LHVCardboard = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of cardboard', 'value'].values[0])
LHVPUfoam = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of polyurethane foam', 'value'].values[0])
LHVglass = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of glass wool', 'value'].values[0])
LHVFoamGlass = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Lower heating value of foam glass', 'value'].values[0])

# Energy recovery efficiency
Xincheat = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Thermal energy recovery efficiency', 'value'].values[0]) 
Xincele = (EndOfLifeInputParameters.loc[EndOfLifeInputParameters['Parameter'] == 'Electrical energy recovery efficiency', 'value'].values[0]) 